---
title: "Object-Oriented Programming"
subtitle: "Inheritance"
author:
  - name: "Mr. OL SAY"
    email: "ol.say@itc.edu.kh"
    affiliations: "Department of Applied Mathematics and Statistics"
institute: "Institute of Technology of Cambodia"
date: "2026-06-30"
date-format: "DD MMMM YYYY"
aspectratio: 169

# =============================================================================
# GENERAL QUARTO & EXECUTION SETTINGS
# =============================================================================
auto-stretch: false
number-sections: true

execute:
  echo: true
  eval: false
  warning: false
  debug: true

# =============================================================================
# COLLAPSE OUTPUT EXTENSION CONFIGURATION
# =============================================================================
extensions:
  collapse-output:
    collapse-all-outputs: true

# =============================================================================
# REVEALJS FORMATTING
# =============================================================================
format:
  revealjs:
    theme: monokai.scss
    mermaid-format: svg # or 'png'
    highlight-style: monokai
    html-math-method: katex
    transition: convex            # Options: none, fade, slide, convex, concave, zoom
    transition-speed: slow       # Options: default, fast, slow
    background-transition: fade  # Transition for background color/images

    # --- Self-Contained Settings (Bundles EVERYTHING) ---
    embed-resources: true # (self-contained: true) 
    self-content-math: true # (or self-contained-math: true)
    include-in-header:
      text: |
        <link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&family=JetBrains+Mono:wght@400;600&display=swap">
        <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@fontsource/stix-two-math@5.3.0/index.css">

    # --- Frame & Scaling Fix ---
    width: 1200
    height: 675
    margin: 0.10
    min-scale: 1.0
    max-scale: 1.0

    # --- Code Display ---
    code-fold: false
    code-summary: "Code"
    code-line-numbers: true

    # --- UI & Slide Content ---
    center: false
    scrollable: true
    footer: "Say OL (D2025504NKH)"
    slide-number: true
---

---

## Inheritance and Class Hierarchies

While **Encapsulation** establishes operational boundaries and protects internal object state by hiding implementation details, **Inheritance** introduces taxonomic hierarchies that model structural relationships between software components. In object-oriented design, inheritance allows a new class-referred to as a **derived class**, **subclass**, or **child class**-to inherit the attributes and methods of an existing class, known as a **base class**, **superclass**, or **parent class**.

---

### Introduction to Base and Derived Classes

In Python, inheritance is declared by passing the base class as an argument within parentheses during class definition.

<style>
  code {
    font-variant-ligatures: none !important;
    font-feature-settings: "liga" 0 !important;
  }
</style>
```python
class BaseDataset:
    def __init__(self, name: str):
        self.name = name

    def get_metadata(self) -> str:
        return f"Dataset Name: {self.name}"

# Derived class inheriting from BaseDataset
class TensorDataset(BaseDataset):
    pass

dataset = TensorDataset("MNIST_Train")
print(dataset.get_metadata())  # Output: Dataset Name: MNIST_Train

print(isinstance(dataset, TensorDataset))      # True
print(isinstance(dataset, BaseDataset))        # True (due to inheritance)
print(issubclass(TensorDataset, BaseDataset))  # True
```

---

### Constructor Extension and the `super()` Function

When a derived class introduces unique attributes, it often needs to define its own `__init__` constructor. However, defining a custom constructor in a subclass completely overrides (shadows) the base class constructor. If left unaddressed, the base class initialization logic is bypassed entirely, leaving inherited attributes unassigned and triggering runtime `AttributeError` exceptions.

```python
class ModelLayer:
    def __init__(self, name: str):
        self.name = name
        self.training = training

class LinearLayer(ModelLayer):
    def __init__(self, name: str, in_features: int, out_features: int):
        # Delegate shared state initialization to the base class
        super().__init__(name, training)
        self.in_features = in_features
        self.out_features = out_features

layer = LinearLayer(name="dense_1", in_features=128, out_features=64)
print(f"Layer Name: {layer.name}")
print(f"Dimensions: {layer.in_features} -> {layer.out_features}")
```

---

### Method Overriding and Behavior Customization

Inheritance does not merely permit code reuse; it enables polymorphism through **method overriding**. When a derived class defines a method with the exact same signature as a method in its base class, the subclass version takes precedence at runtime.

A derived class can choose to completely replace the parent's logic or extend it by invoking the parent method via `super()` and building upon its return value:

```python
class BaseProcessor:
    def process(self, data: list) -> list:
        print("Executing base data cleaning protocol...")
        return [x for x in data if x is not None]

class NormalizingProcessor(BaseProcessor):
    def process(self, data: list) -> list:
        cleaned_data = super().process(data)
        print("Applying standardization scaling...")
        mean = sum(cleaned_data) / len(cleaned_data) if cleaned_data else 0
        return [x - mean for x in cleaned_data]

processor = NormalizingProcessor()
result = processor.process([10, None, 20, 30])
print(f"Processed output: {result}")
```

---

### Inherited Dunder Methods & Protocol Implementation

Inheritance in Python extends far beyond standard attributes and methods. Dunder protocols can be defined at the base class level and inherited, specialized, or enforced across entire class hierarchies.

---

#### Object Representation and Method Overriding (`__str__` and `__repr__`)

By defining string representation methods (`__str__` and `__repr__`) in a base class, derived classes can reuse or override them to provide meaningful, class-specific representations while reducing code duplication.

```python
class Model:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        return f"Model: {self.name}"

    def __repr__(self):
        return f"Model(name='{self.name}')"


class NeuralNetwork(Model):
    def __init__(self, name, layers):
        super().__init__(name)
        self.layers = layers

    def __str__(self):
        return f"Neural Network: {self.name} ({self.layers} layers)"

    def __repr__(self):
        return (
            f"NeuralNetwork(name='{self.name}', "
            f"layers={self.layers})"
        )


model = Model("BaseModel")
network = NeuralNetwork("CNN", 5)

print(str(model))
print(repr(model))

print(str(network))
print(repr(network))
```

---

#### Callable Objects & Template Pattern (`__call__`)

The base class implements the `__call__` dunder method to manage execution infrastructure-such as pre-forward validation-while delegating actual computation to an abstract `forward()` method implemented by derived classes.

```python
class BaseModule:
    def __call__(self, x):
        print(f"[Infrastructure] Running pre-forward hooks for {self.__class__.__name__}")
        if not isinstance(x, list):
            raise TypeError("Input must be a numerical list vector.")
        return self.forward(x)

    def forward(self, x):
        raise NotImplementedError("Derived classes must implement the forward method.")

class ScaleLayer(BaseModule):
    def __init__(self, factor: float):
        self.factor = factor

    def forward(self, x: list) -> list:
        return [val * self.factor for val in x]

layer = ScaleLayer(factor=2.0)
output = layer([1.5, 3.0, 4.5])
print(output)  # [3.0, 6.0, 9.0]
```

---

#### Sequence & Dataset Protocols (`__len__`, `__getitem__`)

By defining `__len__` and `__getitem__` in a base feature container or abstract dataset class, derived subclasses enable native Python indexing, slicing, and iteration.

```python
class BaseDataset:
    def __len__(self) -> int:
        raise NotImplementedError

    def __getitem__(self, index: int):
        raise NotImplementedError

class TabularDataset(BaseDataset):
    def __init__(self, features: list, labels: list):
        if len(features) != len(labels):
            raise ValueError("Feature matrix and label vector lengths must match.")
        self.features = features
        self.labels = labels

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, index: int) -> tuple:
        return self.features[index], self.labels[index]

dataset = TabularDataset(features=[[0.1, 0.2], [0.3, 0.4]], labels=[0, 1])
print(f"Dataset length: {len(dataset)}")
print(f"Sample at index 1: {dataset[1]}")
```

---

#### Multiple Inheritance and Method Resolution Order (MRO)

Python supports Multiple Inheritance, enabling a derived class to inherit behaviors from multiple parent classes simultaneously. Resolution order is determined deterministically via C3 Linearization (`__mro__`).

```python
class Loggable:
    def save(self):
        return "Logging state changes to disk."

class Serializable:
    def save(self):
        return "Serializing object state to JSON."

class DataPipeline(Loggable, Serializable):
    pass

pipeline = DataPipeline()
print(pipeline.save())  # Uses Loggable due to MRO ordering
print(DataPipeline.__mro__)
```

---

### Case Study: Mini-Pandas Architecture (`Series` and `DataFrame`)

```python
class BaseContainer:
    """Base container providing shared length and representation protocols."""
    def __len__(self) -> int:
        raise NotImplementedError("Subclasses must implement __len__.")

    def __repr__(self) -> str:
        return f"<{self.__class__.__name__}>"

class Series(BaseContainer):
    def __init__(self, data: list, index: list = None, name: str = "Series"):
        self.data = data
        self.index = index if index is not None else list(range(len(data)))
        self.name = name
        
        if len(self.data) != len(self.index):
            raise ValueError("Data and index lengths must match.")

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, key):
        if key in self.index:
            return self.data[self.index.index(key)]
        elif isinstance(key, int):
            return self.data[key]
        raise KeyError(f"Key '{key}' not found in Series index.")

    def __repr__(self) -> str:
        mapping = dict(zip(self.index, self.data))
        return f"Series(name='{self.name}', values={mapping})"

class DataFrame(BaseContainer):
    def __init__(self, data: dict[str, list], index: list = None):
        self.columns = list(data.keys())
        sample_col = next(iter(data.values())) if data else []
        self._index = index if index is not None else list(range(len(sample_col)))
        
        self._series_map = {
            col: Series(vals, index=self._index, name=col) 
            for col, vals in data.items()
        }

    def __len__(self) -> int:
        return len(self._index)

    def __getitem__(self, key: str) -> Series:
        if not isinstance(key, str):
            raise TypeError("DataFrame column lookup requires a string key.")
        if key not in self._series_map:
            raise KeyError(f"Column '{key}' not found in DataFrame.")
        return self._series_map[key]

    def __repr__(self) -> str:
        col_summary = ", ".join(f"'{col}'" for col in self.columns)
        return f"DataFrame(shape=({len(self)} rows, {len(self.columns)} cols), columns=[{col_summary}])"

```

---

### Execution and Demonstration

```python
#| label: demonstrate-mini-pandas-pipeline
salaries = Series([85000, 115000, 130000], index=["Alice", "Bob", "Charlie"], name="Salary")
print(salaries)
print(f"Salary lookup for Bob: {salaries['Bob']}")

table_data = {
    "Department": ["HR", "Engineering", "Research"],
    "Salary": [85000, 115000, 130000],
    "Remote": [False, True, True]
}
df = DataFrame(table_data, index=["Alice", "Bob", "Charlie"])
print(df)

dept_series = df["Department"]
print(dept_series)
print(f"Department for Charlie: {dept_series['Charlie']}")
```

---

### Practical Exercises

::: {.callout-tip title="Exercise 1: Multi-Level Constructor Chaining in Configuration Hierarchies" icon=false}
**Objective:** Design a three-tier configuration hierarchy for machine learning pipelines to practice multi-level `super()` delegation.

**Architectural Context:** When building complex software, subclasses often need to initialize attributes belonging to their parent and grandparent classes without duplicating code. Using `super()` correctly ensures control flows smoothly up the inheritance chain.

**Step-by-Step Instructions:**

1. **Create the Base Class (`BaseConfig`)**: Define an `__init__` method that accepts `project_name: str` and stores it as an instance attribute.
2. **Create the Intermediate Class (`SupervisedConfig`)**: Inherit from `BaseConfig`. Its `__init__` should accept both `project_name` and an additional parameter `random_seed: int = 42`. Use `super().__init__(project_name)` to pass the project name up to the base class.
3. **Create the Final Derived Class (`TransformerConfig`)**: Inherit from `SupervisedConfig`. Its `__init__` should accept `project_name`, `random_seed`, and `max_seq_length: int = 512`. Chain initialization upward through the entire hierarchy using `super().__init__(project_name, random_seed)`.
4. **Verification**: Instantiate `TransformerConfig(project_name="NLP-Pipeline", random_seed=123, max_seq_length=256)` and verify that all three attributes are correctly bound on the final instance.
:::


---


::: {.callout-tip title="Exercise 2: Template Method Pattern via `__call__` and Method Overriding" icon=false}
**Objective:** Implement a reusable data transformation pipeline using the template method design pattern.

**Architectural Context:** The template method pattern defines the skeleton of an algorithm in a base class while allowing subclasses to override specific steps without changing the overall execution flow.

**Step-by-Step Instructions:**

1. **Build the Base Class (`BaseTransformer`)**: Implement the `__call__(self, data: list)` dunder method. Inside `__call__`, print a tracking log: `f"Executing {self.__class__.__name__} pipeline..."` and return the result of calling `self.transform(data)`. The base `transform` method should raise a `NotImplementedError` to enforce implementation by child classes.
2. **Build the Derived Class (`MinMaxNormalizer`)**: Create a child class inheriting from `BaseTransformer`. Override the `transform(self, data: list)` method to scale all numeric values in the list between `0.0` and `1.0` using the formula: $(x - \text{min}) / (\text{max} - \text{min})$.
3. **Test the Pipeline**: Instantiate `MinMaxNormalizer`, and invoke the instance directly as a function (e.g., `normalizer([10.0, 20.0, 50.0])`), confirming that both the base infrastructure log and child transformation logic execute sequentially.
:::


---


::: {.callout-tip title="Exercise 3: Mixin-Based Multiple Inheritance for Experiment Tracking" icon=false}
**Objective:** Build a modular experiment logger using independent mixin classes and multiple inheritance.

**Architectural Context:** Mixins are small, focused classes designed to provide specific auxiliary behaviors to other classes via multiple inheritance without acting as standalone data models.

**Step-by-Step Instructions:**

1. **Create the Timestamp Mixin (`TimestampMixin`)**: Define an `__init__` method (using cooperative `super().__init__()`) that assigns a `created_at` timestamp string (e.g., using `datetime.now().isoformat()`) to the instance.
2. **Create the Dictionary Export Mixin (`DictionaryExportMixin`)**: Add a method `to_dict(self)` that returns `self.__dict__`.
3. **Create the Base Experiment (`BaseExperiment`)**: Define an `__init__` that stores an `experiment_id: str` and correctly calls `super().__init__()`.
4. **Create the Tracked Experiment (`TrackedExperiment`)**: Inherit from `BaseExperiment`, `TimestampMixin`, and `DictionaryExportMixin`. Instantiate it with an ID and verify that calling `.to_dict()` successfully outputs the combined attributes from all parents.
:::

---

::: {.callout-tip title="Exercise 04: Encapsulated Base Model with Protected State and Validation" icon=false}
**Objective:** Combine encapsulation (private/protected attributes and property getters/setters) with class inheritance and method overriding.

**Architectural Context:** In machine learning pipelines, base models often require strict validation on internal parameters (like learning rates or weights) to prevent invalid states. Encapsulation protects these variables, while inheritance allows specialized models to extend or override model behavior.

**Step-by-Step Instructions:**

1. **Create an Encapsulated Base Model (`BaseModel`)**:
   - Define an `__init__` taking `model_name: str` and a private attribute `__learning_rate: float`.
   - Implement a `@property` getter and a `@property` setter for `learning_rate` that validates the input (e.g., raising a `ValueError` if the learning rate is $\le 0$ or $> 1.0$).
   - Add a method `train(self, data: list) -> str` that returns a generic training status string.
2. **Create a Specialized Derived Model (`DeepLearningModel`)**:
   - Inherit from `BaseModel`. Add an additional protected attribute `_epochs: int`.
   - Override the `train(self, data: list)` method to include epochs and the encapsulated learning rate in its return string (e.g., accessing `self.learning_rate` safely through the inherited property).
3. **Test and Verify**: Instantiate `DeepLearningModel`, attempt to set an invalid learning rate (e.g., `1.5`) to verify that the encapsulated validation catches it, and then execute the `.train()` method.
:::


---


::: {.callout-tip title="Exercise 05: Secure Multi-Tier Data Pipeline with Encapsulated Credentials" icon=false}
**Objective:** Implement a multi-level inheritance hierarchy where base classes manage secure encapsulated configuration state and derived classes execute operational logic.

**Architectural Context:** Multi-level inheritance paired with encapsulation allows infrastructure security rules (like masked API tokens or read-only connection strings) to be enforced at the root level and inherited across specialized processing pipelines.

**Step-by-Step Instructions:**

1. **Create Root Secure Configuration (`SecureConfigBase`)**:
   - Accept a sensitive string `api_token: str`. Encapsulate it using a private attribute `__api_token`.
   - Provide a getter property `api_token` that returns a masked version of the token (e.g., displaying only the last 4 characters, like `****1234`) for logging safety.
   - Provide a method `verify_access(self, token: str) -> bool` to securely check input tokens against the private attribute.
2. **Create an Intermediate Pipeline Class (`PipelineBase`)**:
   - Inherit from `SecureConfigBase`, accepting `pipeline_name: str` and `api_token`. Chain initialization using `super()`.
   - Add a protected attribute `_status: str = "Idle"`.
3. **Create the Final Executable Pipeline (`CloudIngestPipeline`)**:
   - Inherit from `PipelineBase`. Implement a method `run_ingest(self, source: str) -> None` that checks access using inherited security methods, updates `_status` to `"Running"`, and prints an execution summary using the masked token.
4. **Test the System**: Instantiate `CloudIngestPipeline`, verify that direct access to the raw private token fails (`AttributeError`), test the masked property, and execute `run_ingest()`.
:::